In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

####1. Create a Streaming Dataframe

In [0]:
from pyspark.sql.functions import col, current_timestamp, expr

stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load() \
    .select(
        col("value").cast("int").alias("event_id"),
        expr("CASE WHEN value % 3 = 0 THEN 'purchase' "
             "     WHEN value % 3 = 1 THEN 'click' "
             "     ELSE 'view' END").alias("event_type"),
        expr("cast(500 + (value % 100) as int)").alias("user_id"),
        current_timestamp().alias("event_ts")
    )


####2. Start the streaming write

In [0]:
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/temp/checkpoints/events_stream/") \
    .trigger(processingTime="5 seconds") \
    .toTable("events")

print("Streaming query started. Writing to main.default.events every 5 seconds.")

####3. First batch read while stream is running
[WAIT at least 10 seconds after previous Cell finishes, then RUN this Cell]

In [0]:
count_1 = spark.table("events").count()
print(f"Row count (read 1): {count_1}")

####4. Wait, then read again
[WAIT 15 seconds, then RUN this Cell]

In [0]:
count_2 = spark.table("events").count()
print(f"Row count (read 2): {count_2}")
print(f"New rows landed by stream: {count_2 - count_1}")

> The count is higher. \
The stream wrote new rows while we were waiting, and the batch query is seeing them immediately — no cache to invalidate, no delay, no coordination. Every micro-batch commits becomes instantly visible to any batch reader.\
This is the unified table in action. The same Delta table, the same S3 files, the same transaction log — serving a streaming writer and a batch reader at the same time, safely.

####5. Stop the stream

In [0]:
query.stop()
print("Stream stopped.")

####6.  Final table state

In [0]:
final_count = spark.table("events").count()
print(f"Final row count: {final_count}")

In [0]:
%sql
DESCRIBE HISTORY events;